In [1]:
# Ячейка 1: Импорты
import sys
from pathlib import Path
import json
from datetime import datetime
from IPython.display import display, HTML, Markdown
import pandas as pd

# Добавляем путь к проекту
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"📁 Project root: {PROJECT_ROOT}")
print(f"✅ Python path configured")

📁 Project root: d:\Work\plagin\js_plagin
✅ Python path configured


In [2]:
# Ячейка 2: Проверка подключений к базам данных
from src.backend.utils.configs import Config
from qdrant_client import QdrantClient
from sqlalchemy import create_engine, text

config = Config.load_config()

print("="*60)
print("🔍 CHECKING DATABASE CONNECTIONS")
print("="*60)

# Проверка SQL
try:
    sql_url = f"mysql+aiomysql://{config.db.USER}:{config.db.PASSWORD.get_secret_value()}@{config.db.HOST}:{config.db.PORT}/{config.db.NAME}"
    # Для синхронного подключения используем pymysql
    sql_url_sync = sql_url.replace('aiomysql', 'pymysql')
    engine = create_engine(sql_url_sync, echo=False)
    
    with engine.connect() as conn:
        result = conn.execute(text("SELECT COUNT(*) as count FROM meet"))
        count = result.fetchone()[0]
    
    print(f"\n✅ SQL Connection: OK")
    print(f"   Host: {config.db.HOST}:{config.db.PORT}")
    print(f"   Database: {config.db.NAME}")
    print(f"   Meetings in DB: {count}")
    
except Exception as e:
    print(f"\n❌ SQL Connection: FAILED")
    print(f"   Error: {e}")

# Проверка Qdrant
try:
    qdrant_params = {"url": config.vectordb.URL, "timeout": 10}
    if config.vectordb.API_KEY:
        qdrant_params["api_key"] = config.vectordb.API_KEY.get_secret_value()
    
    qdrant_client = QdrantClient(**qdrant_params)
    collections = qdrant_client.get_collections()
    
    print(f"\n✅ Qdrant Connection: OK")
    print(f"   URL: {config.vectordb.URL}")
    print(f"   Collections: {len(collections.collections)}")
    
    for coll in collections.collections:
        print(f"      - {coll.name}")
    
except Exception as e:
    print(f"\n❌ Qdrant Connection: FAILED")
    print(f"   Error: {e}")

print("\n" + "="*60)

🔍 CHECKING DATABASE CONNECTIONS

✅ SQL Connection: OK
   Host: 168.231.125.115:3306
   Database: Qontex
   Meetings in DB: 83

✅ Qdrant Connection: OK
   URL: http://localhost:6333
   Collections: 3
      - client_profiles
      - laws
      - meetings



In [3]:
# Ячейка 3: Инициализация SQLQdrantSynchronizer
from src.backend.vector_db.sql_to_vector import SQLQdrantSynchronizer

print("🚀 Initializing SQL to Qdrant Synchronizer...")
print("-"*60)

try:
    syncer = SQLQdrantSynchronizer()
    print("✅ Synchronizer initialized successfully!")
    
except Exception as e:
    print(f"❌ Failed to initialize synchronizer: {e}")
    raise

🚀 Initializing SQL to Qdrant Synchronizer...
------------------------------------------------------------


[2025-10-02 20:16:10] [INFO] SQL connection test: OK
[2025-10-02 20:16:10] [INFO] Qdrant connection test: OK
[2025-10-02 20:16:10] [INFO] Collection 'meetings' already exists
[2025-10-02 20:16:10] [INFO] Initialized with table: meet
[2025-10-02 20:16:10] [INFO] SQL Host: 168.231.125.115:3306
[2025-10-02 20:16:10] [INFO] Qdrant URL: http://localhost:6333


✅ Synchronizer initialized successfully!


In [4]:
# Ячейка 4: Проверка состояния коллекции meetings
print("="*60)
print("📊 CURRENT STATE OF 'meetings' COLLECTION")
print("="*60)

try:
    collection_info = syncer.qdrant_client.get_collection("meetings")
    
    print(f"\n✅ Collection exists")
    print(f"   Name: {collection_info.config.params.vectors.size}")
    print(f"   Vectors count: {collection_info.vectors_count:,}")
    print(f"   Points count: {collection_info.points_count:,}")
    print(f"   Status: {collection_info.status}")
    print(f"   Vector size: {collection_info.config.params.vectors.size}")
    print(f"   Distance metric: {collection_info.config.params.vectors.distance}")
    
    # Получаем примеры данных
    sample = syncer.qdrant_client.scroll(
        collection_name="meetings",
        limit=3,
        with_payload=True,
        with_vectors=False
    )
    
    if sample[0]:
        print(f"\n📝 Sample data (showing {len(sample[0])} points):")
        for i, point in enumerate(sample[0], 1):
            print(f"\n   Point {i}:")
            print(f"      ID: {point.id}")
            print(f"      Meeting ID: {point.payload.get('meeting_id', 'N/A')}")
            print(f"      Title: {point.payload.get('title', 'N/A')[:50]}...")
            print(f"      Chunk: {point.payload.get('chunk', 'N/A')}/{point.payload.get('total_chunks', 'N/A')}")
            print(f"      Date: {point.payload.get('date_start', 'N/A')}")
    
except Exception as e:
    print(f"⚠️  Collection 'meetings' does not exist or error: {e}")
    print("   This is OK if you haven't synced data yet.")

print("\n" + "="*60)

📊 CURRENT STATE OF 'meetings' COLLECTION

✅ Collection exists
   Name: 1536
⚠️  Collection 'meetings' does not exist or error: unsupported format string passed to NoneType.__format__
   This is OK if you haven't synced data yet.



In [4]:
import requests
import json
from datetime import datetime
from IPython.display import display, JSON, Markdown

BASE_URL = "http://localhost:8000"

def print_json(data):
  
    display(JSON(data))

def print_md(text):

    display(Markdown(text))

In [7]:
 
payload = {
    "message": "About What were last two meets?"
}

print("Question:")
print_json(payload)

response = requests.post(f"{BASE_URL}/rag_chat", json=payload)

print("\nAnswer:")
data = response.json()
print_json(data)
 

Question:


<IPython.core.display.JSON object>


Answer:


<IPython.core.display.JSON object>

In [9]:
print("=== Sync all meetings ===")
response = requests.post(f"{BASE_URL}/sql_to_vector")
print(f"Status: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}\n")

=== Sync all meetings ===
Status: 200
Response: {
  "ok": true,
  "processed": 83,
  "added": 0,
  "skipped": 30,
  "errors": 53
}



In [8]:
print(json.dumps(data, indent=2, ensure_ascii=False))

[
  "467b3365-d88a-435f-a602-58562a0f7c06",
  "9bd1981d-6c70-4136-9dc7-82cb3e108f28",
  {
    "message": "Based on the provided meeting fragments, the last two meetings were as follows:\n\n---\n\n- **Meeting 1 (Most Recent):**\n  - **Title:** Test meet\n  - **Date:** 2025-10-02T19:37:12\n  - **Meeting ID:** b0c19ac9-2852-42c6-876c-4c6208f27c84\n  - **Language:** ru\n  - **Participant:** Jora Nosorog\n  - **Key Content:**\n    - Discussion centered on violations related to the Ministry of Health:  \n      > \"Я нарушил закон, связанный с Министерством здравоохранения.\"\n    - Repeated mention of health law violations and the topic of cancer:  \n      > \"Опять, я нарушил закон, связанный с здравоохранением, прием, как слышно.\"  \n      > \"Рак. Рак. Тут должен быть рак. Но рака нет.\"\n    - The meeting included a comprehensive financial advisor report, covering:\n      - Client profile (age 33, unmarried, employed as Marketing Specialist, income $92,000/year)\n      - Financial posit